In [4]:
from typing import Any, Dict, List, Optional, Iterator
import olefile
import zlib
import struct
import re
import unicodedata
from langchain_core.documents import Document
from langchain_core.document_loaders import BaseLoader


class HWPLoader(BaseLoader):
    """HWP 파일 읽기 클래스. HWP 파일의 내용을 읽습니다."""

    def __init__(self, file_path: str, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.file_path = file_path
        self.extra_info = {"source": file_path}
        self._initialize_constants()

    def _initialize_constants(self) -> None:
        """상수 초기화 메서드"""
        self.FILE_HEADER_SECTION = "FileHeader"
        self.HWP_SUMMARY_SECTION = "\x05HwpSummaryInformation"
        self.SECTION_NAME_LENGTH = len("Section")
        self.BODYTEXT_SECTION = "BodyText"
        self.HWP_TEXT_TAGS = [67]

    def lazy_load(self) -> Iterator[Document]:
        """HWP 파일에서 데이터를 로드하고 표를 추출합니다.

        Yields:
            Document: 추출된 문서
        """
        load_file = olefile.OleFileIO(self.file_path)
        file_dir = load_file.listdir()

        if not self._is_valid_hwp(file_dir):
            raise ValueError("유효하지 않은 HWP 파일입니다.")

        result_text = self._extract_text(load_file, file_dir)
        yield self._create_document(text=result_text, extra_info=self.extra_info)

    def _is_valid_hwp(self, dirs: List[List[str]]) -> bool:
        """HWP 파일의 유효성을 검사합니다."""
        return [self.FILE_HEADER_SECTION] in dirs and [self.HWP_SUMMARY_SECTION] in dirs

    def _get_body_sections(self, dirs: List[List[str]]) -> List[str]:
        """본문 섹션 목록을 반환합니다."""
        section_numbers = [
            int(d[1][self.SECTION_NAME_LENGTH :])
            for d in dirs
            if d[0] == self.BODYTEXT_SECTION
        ]
        return [
            f"{self.BODYTEXT_SECTION}/Section{num}" for num in sorted(section_numbers)
        ]

    def _create_document(
        self, text: str, extra_info: Optional[Dict] = None
    ) -> Document:
        """문서 객체를 생성합니다."""
        return Document(page_content=text, metadata=extra_info or {})

    def _extract_text(
        self, load_file: olefile.OleFileIO, file_dir: List[List[str]]
    ) -> str:
        """모든 섹션에서 텍스트를 추출합니다."""
        sections = self._get_body_sections(file_dir)
        return "\n".join(
            self._get_text_from_section(load_file, section) for section in sections
        )

    def _is_compressed(self, load_file: olefile.OleFileIO) -> bool:
        """파일이 압축되었는지 확인합니다."""
        with load_file.openstream(self.FILE_HEADER_SECTION) as header:
            header_data = header.read()
            return bool(header_data[36] & 1)

    def _get_text_from_section(self, load_file: olefile.OleFileIO, section: str) -> str:
        """특정 섹션에서 텍스트를 추출합니다."""
        with load_file.openstream(section) as bodytext:
            data = bodytext.read()

        unpacked_data = (
            zlib.decompress(data, -15) if self._is_compressed(load_file) else data
        )

        text = []
        i = 0
        while i < len(unpacked_data):
            header, rec_type, rec_len = self._parse_record_header(
                unpacked_data[i : i + 4]
            )
            if rec_type in self.HWP_TEXT_TAGS:
                rec_data = unpacked_data[i + 4 : i + 4 + rec_len]
                text.append(rec_data.decode("utf-16"))
            i += 4 + rec_len

        text = "\n".join(text)
        text = self.remove_chinese_characters(text)
        text = self.remove_control_characters(text)
        return text

    @staticmethod
    def remove_chinese_characters(s: str):
        """중국어 문자를 제거합니다."""
        return re.sub(r"[\u4e00-\u9fff]+", "", s)

    @staticmethod
    def remove_control_characters(s):
        """깨지는 문자 제거"""
        return "".join(ch for ch in s if unicodedata.category(ch)[0] != "C")

    @staticmethod
    def _parse_record_header(header_bytes: bytes) -> tuple:
        """레코드 헤더를 파싱합니다."""
        header = struct.unpack_from("<I", header_bytes)[0]
        rec_type = header & 0x3FF
        rec_len = (header >> 20) & 0xFFF
        return header, rec_type, rec_len


In [5]:
loader = HWPLoader("../data/디지털 정부혁신 추진계획.hwp")

docs = loader.load()
print(docs[0].page_content[:1000])

디지털 정부혁신 추진계획2019. 10. 29.      관계부처 합동순    서Ⅰ. 개요ȃ 1Ⅱ. 디지털 정부혁신 추진계획ㆬȃ 2  1. 우선 추진과제ȃ 2     ① 선제적·통합적 대국민 서비스 혁신     ② 공공부문 마이데이터 활성화     ③ 시민참여를 위한 플랫폼 고도화     ④ 현장중심 협업을 지원하는 스마트 업무환경 구현     ⑤ 클라우드와 디지털서비스 이용 활성화     ⑥ 개방형 데이터·서비스 생태계 구축  2. 중장기 범정부 디지털 전환 로드맵 수립ᲈȃ 4Ⅲ. 추진체계 및 일정ȃ 4 [붙임] 디지털 정부혁신 우선 추진과제(상세)ᬜȃ 8Ⅰ. 개 요□ 추진 배경 ○ 우리나라는 국가적 초고속 정보통신망 투자와 적극적인 공공정보화 사업 추진에 힘입어 세계 최고수준의 전자정부를 구축‧운영     * UN전자정부평가에서 2010‧12‧14년 1위, 16‧18년 3위, UN공공행정상 13회 수상 ○ 그러나, 인공지능‧클라우드 중심의 디지털 전환(Digital Transformation) 시대가 도래함에 따라 기존 전자정부의 한계 표출   - 축적된 행정데이터에도 불구하고 기관간 연계‧활용 미흡, 부처 단위로 단절된 서비스, 신기술 활용을 위한 제도‧기반 부족   - 디지털 전환을 위한 컨트롤타워가 없고, 구체적 전략도 부재 ○ 이에, ‘19.3월부터 공공부문 ICT 활용현황 및 문제점 검토에 착수하여 공공분야 디지털 전환을 위한 추진계획 마련     * 관계부처 협의 21회(행안,과기정통,기재,복지,권익위,국정원 등), 민간전문가 의견청취 10회□ 문제점 진단 및 평가 ○ (서비스) 국민과 최종 이용자 관점에서 서비스 혁신 미흡   - 자격이 있어도 자신이 받을 수 있는 공공서비스를 파악하기 어려워 사각지대가 발생하고, 온라인 신청 가능한 서비스도 제한적 ○ (데이터) 기관별로 축적·보유한 데이터의 연계와 활용 부족   - A기관에서 서류를 발급받아 B기관에 제출하는 관행(연간 증명서 9.5억건‘18년 발급) 등 데이터가 국민편익 향상에 제대로 활용